In [ ]:
#x-y平面に流線を描画するコード（jeans_3d-test26の可視化で使った）

import os
import numpy as np
import pyvista as pv
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import natsort
import matplotlib.animation as animation
from PIL import Image

vtk_dir = os.path.expanduser("~/athenapp/results/〇〇")
output_dir = "./xy_streamline"
os.makedirs(output_dir, exist_ok=True)

# 計算領域
Lbox = 12.0
x_min, x_max = 0.0, 12.0
y_min, y_max = 0.0, 12.0

# 球
Rsphere = 5.0
xc, yc = 6.0, 6.0

# 中央スライス
z_slice = 6.0

# ============================================================
# VTK ファイル取得
# ============================================================
vtk_files = natsort.natsorted([
    os.path.join(vtk_dir, f)
    for f in os.listdir(vtk_dir)
    if f.startswith("Jeans.block") and f.endswith(".vtk")
])

block_dict = {}
for f in vtk_files:
    block_id = int(os.path.basename(f).split('.')[1].replace('block', ''))
    block_dict.setdefault(block_id, []).append(f)

all_blocks_sorted = [block_dict[b] for b in sorted(block_dict.keys())]

# ============================================================
# 球内のみで密度スケール決定
# ============================================================
dens_inside_all = []

for step_files in zip(*all_blocks_sorted):
    pts_all, dens_all = [], []

    for f in step_files:
        grid = pv.read(f)
        sl = grid.slice(normal='z', origin=(0, 0, z_slice))
        pts_all.append(sl.cell_centers().points)
        dens_all.append(sl['dens'])

    pts_all = np.vstack(pts_all)
    dens_all = np.hstack(dens_all)

    r = np.sqrt((pts_all[:, 0]-xc)**2 + (pts_all[:, 1]-yc)**2)
    dens_inside_all.append(dens_all[r <= Rsphere])

dens_inside_all = np.concatenate(dens_inside_all)
vmin = np.percentile(dens_inside_all, 5)
vmax = np.percentile(dens_inside_all, 95)

# ============================================================
# 各タイムステップ描画
# ============================================================
for step_files in zip(*all_blocks_sorted):

    pts_all, dens_all, vx_all, vy_all = [], [], [], []

    for f in step_files:
        grid = pv.read(f)
        sl = grid.slice(normal='z', origin=(0, 0, z_slice))

        centers = sl.cell_centers().points
        dens = sl['dens']
        mom  = sl['mom']

        pts_all.append(centers)
        dens_all.append(dens)
        vx_all.append(mom[:, 0] / dens)
        vy_all.append(mom[:, 1] / dens)

    pts_all = np.vstack(pts_all)
    dens_all = np.hstack(dens_all)
    vx_all = np.hstack(vx_all)
    vy_all = np.hstack(vy_all)

    xs = np.unique(pts_all[:, 0])
    ys = np.unique(pts_all[:, 1])
    nx, ny = len(xs), len(ys)

    dens_2d = np.full((ny, nx), np.nan)
    vx_2d   = np.full((ny, nx), np.nan)
    vy_2d   = np.full((ny, nx), np.nan)

    for i, x in enumerate(xs):
        for j, y in enumerate(ys):
            m = (np.isclose(pts_all[:, 0], x)) & (np.isclose(pts_all[:, 1], y))
            if np.any(m):
                dens_2d[j, i] = dens_all[m][0]
                vx_2d[j, i]   = vx_all[m][0]
                vy_2d[j, i]   = vy_all[m][0]

    X, Y = np.meshgrid(xs, ys)
    r = np.sqrt((X-xc)**2 + (Y-yc)**2)

    dens_in  = np.ma.masked_where(r > Rsphere, dens_2d)
    dens_out = np.ma.masked_where(r <= Rsphere, dens_2d)

    # ========================================================
    # 描画
    # ========================================================
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.set_aspect('equal')

    # 球外
    ax.pcolormesh(xs, ys, dens_out, color='#0b3c5d')

    # 球内密度
    im = ax.pcolormesh(xs, ys, dens_in,
                       cmap='inferno',
                       norm=LogNorm(vmin=vmin, vmax=vmax))

    # ===== 流線 =====
    ax.streamplot(
        xs, ys,
        vx_2d, vy_2d,
        color='white',
        density=1.2,
        linewidth=0.7,
        arrowsize=0.8
    )

    # 球境界
    theta = np.linspace(0, 2*np.pi, 400)
    ax.plot(xc + Rsphere*np.cos(theta),
            yc + Rsphere*np.sin(theta),
            color='cyan', lw=0.6)

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")

    filename = os.path.basename(step_files[0])
    num = filename.split('.')[-2]
    ax.set_title(f"Streamlines (γ=1.2, β=0.05)")

    fig.colorbar(im, ax=ax, label="Density (inside sphere)")

    png_file = os.path.join(output_dir, f"stream_{num}.png")
    fig.savefig(png_file, dpi=150, bbox_inches='tight')
    plt.close(fig)

    print(f"Saved {png_file}")

In [ ]:
#Toyouchi-test12を可視化するときに使った、流線on密度mapを作成するコード
#Toyouchi+23の再現は今後これを使用すると良い

import os
import numpy as np
import pyvista as pv
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import natsort
from scipy.interpolate import griddata

# ============================================================
# ユーザー設定
# ============================================================
vtk_dir = os.path.expanduser("~/athena-project/results/〇〇")  # 適宜変更
output_dir = "./x-y_streamline"
os.makedirs(output_dir, exist_ok=True)

# 計算領域（code unit）
x_min, x_max = -224, 224
y_min, y_max = -224, 224

# スライスするz座標（2D計算なら0.0でOK）
z_slice = 0.0

# 可視化する変数（Athena++の出力に合わせる）
var_name = 'rho'      # 密度

# 流線の密度（数値を大きくすると流線が増える）
stream_density = 1.5
stream_linewidth = 0.35   # 線幅
stream_color = 'white'    # 流線の色

# カラーマップの範囲（自動設定する場合はNone）
vmin_user = None
vmax_user = None

# 内挿グリッドの解像度（数値を大きくすると高解像度）
grid_resolution = 200  # 200x200の均一格子上に内挿

# ============================================================
# VTKファイルの読み込み
# ============================================================
vtk_files = natsort.natsorted([
    os.path.join(vtk_dir, f)
    for f in os.listdir(vtk_dir)
    if f.startswith("Toyouchi.block") and f.endswith(".vtk")
])

print(f"Found {len(vtk_files)} VTK files")

# ブロックごとにグループ化
block_dict = {}
for f in vtk_files:
    block_id = int(os.path.basename(f).split('.')[1].replace('block', ''))
    block_dict.setdefault(block_id, []).append(f)

all_blocks_sorted = [block_dict[b] for b in sorted(block_dict.keys())]
print(f"Number of blocks: {len(all_blocks_sorted)}")
print(f"Number of time steps: {len(all_blocks_sorted[0]) if all_blocks_sorted else 0}")

if not all_blocks_sorted:
    print("No VTK files found. Check the directory path.")
    exit()

# ============================================================
# 密度スケールを自動決定（全タイムステップから）
# ============================================================
if vmin_user is None or vmax_user is None:
    var_all_steps = []
    for step_idx, step_files in enumerate(zip(*all_blocks_sorted)):
        print(f"Scanning time step {step_idx+1}/{len(all_blocks_sorted[0])} for color scale...")
        for f in step_files:
            grid = pv.read(f)
            sl = grid.slice(normal='z', origin=(0, 0, z_slice))
            var_all_steps.append(sl[var_name])
    
    var_all_steps = np.hstack(var_all_steps)
    vmin = vmin_user if vmin_user else np.percentile(var_all_steps, 2)
    vmax = vmax_user if vmax_user else np.percentile(var_all_steps, 98)
else:
    vmin, vmax = vmin_user, vmax_user

print(f"Color scale: vmin={vmin:.3e}, vmax={vmax:.3e}")

# ============================================================
# 各タイムステップで描画
# ============================================================
for step_idx, step_files in enumerate(zip(*all_blocks_sorted)):
    print(f"Processing time step {step_idx+1}/{len(all_blocks_sorted[0])}...")

    pts_all, var_all, vx_all, vy_all = [], [], [], []

    for f in step_files:
        grid = pv.read(f)
        sl = grid.slice(normal='z', origin=(0, 0, z_slice))

        centers = sl.cell_centers().points
        var = sl['rho']
        vel = sl['vel']
        vx = vel[:, 0]
        vy = vel[:, 1]

        pts_all.append(centers)
        var_all.append(var)
        vx_all.append(vx)
        vy_all.append(vy)

    pts_all = np.vstack(pts_all)
    var_all = np.hstack(var_all)
    vx_all = np.hstack(vx_all)
    vy_all = np.hstack(vy_all)

    # 計算領域内の点のみを使用
    mask = (pts_all[:, 0] >= x_min) & (pts_all[:, 0] <= x_max) & \
           (pts_all[:, 1] >= y_min) & (pts_all[:, 1] <= y_max)
    
    pts_all = pts_all[mask]
    var_all = var_all[mask]
    vx_all = vx_all[mask]
    vy_all = vy_all[mask]

    # 等間隔なグリッドを作成
    xi = np.linspace(x_min, x_max, grid_resolution)
    yi = np.linspace(y_min, y_max, grid_resolution)
    X, Y = np.meshgrid(xi, yi)

    # 2次元の点群として内挿（x, y座標のみ使用）
    print("  Interpolating data to regular grid...")
    points_2d = pts_all[:, :2]  # x, y座標のみ抽出
    
    try:
        # method='linear'で内挿
        var_2d = griddata(points_2d, var_all, (X, Y), method='linear', fill_value=np.nan)
        vx_2d = griddata(points_2d, vx_all, (X, Y), method='linear', fill_value=np.nan)
        vy_2d = griddata(points_2d, vy_all, (X, Y), method='linear', fill_value=np.nan)
    except Exception as e:
        print(f"  Linear interpolation failed: {e}")
        print("  Falling back to nearest interpolation...")
        # フォールバック：nearest法を使用
        var_2d = griddata(points_2d, var_all, (X, Y), method='nearest', fill_value=np.nan)
        vx_2d = griddata(points_2d, vx_all, (X, Y), method='nearest', fill_value=np.nan)
        vy_2d = griddata(points_2d, vy_all, (X, Y), method='nearest', fill_value=np.nan)

    # NaNをマスク（領域外のデータを非表示に）
    var_2d = np.ma.masked_invalid(var_2d)
    vx_2d = np.ma.masked_invalid(vx_2d)
    vy_2d = np.ma.masked_invalid(vy_2d)

    # データがすべてNaNの場合のチェック
    if np.all(var_2d.mask):
        print(f"  Warning: No valid data after interpolation for step {step_idx}")
        continue

    # ========================================================
    # 描画
    # ========================================================
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_aspect('equal')

    # 密度分布（対数スケール）
    try:
        im = ax.pcolormesh(X, Y, var_2d,
                           cmap='plasma',
                           norm=LogNorm(vmin=vmin, vmax=vmax),
                           shading='auto')
    except Exception as e:
        print(f"  pcolormesh failed: {e}")
        # フォールバック：imshowを使用
        im = ax.imshow(var_2d.T, origin='lower', extent=[x_min, x_max, y_min, y_max],
                       cmap='plasma', norm=LogNorm(vmin=vmin, vmax=vmax))
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_min, y_max)

    # ===== 流線 =====
    # 速度場の極端な値をクリップ
    vmag = np.sqrt(vx_2d**2 + vy_2d**2)
    valid_vmag = vmag[~np.ma.masked_invalid(vmag).mask]
    
    if len(valid_vmag) > 0:
        vmax_clip = np.percentile(valid_vmag, 99)
        
        vx_clipped = np.clip(vx_2d.filled(0), -vmax_clip, vmax_clip)
        vy_clipped = np.clip(vy_2d.filled(0), -vmax_clip, vmax_clip)
        
        # 流線描画
        try:
            ax.streamplot(
                xi, yi,
                vx_clipped, vy_clipped,
                color=stream_color,
                density=stream_density,
                linewidth=stream_linewidth,
                arrowsize=0.6,
                arrowstyle='->',
                minlength=0.1,
                broken_streamlines=False
            )
        except Exception as e:
            print(f"  Streamplot failed: {e}")
    else:
        print("  Warning: No valid velocity data for streamlines")

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel("X (×450AU)", fontsize=12)
    ax.set_ylabel("Y (×450AU)", fontsize=12)

    # タイムステップ情報
    ax.set_title(f"Density & Streamlines (t = step {step_idx})", fontsize=12)

    cbar = fig.colorbar(im, ax=ax, label=f"Density (code unit)")
    cbar.ax.tick_params(labelsize=10)

    png_file = os.path.join(output_dir, f"stream_{step_idx:04d}.png")
    fig.savefig(png_file, dpi=150, bbox_inches='tight')
    plt.close(fig)

    print(f"Saved {png_file}")

print("All processing completed!")